# View the arithmetic datasets

Load, filter, and inspect the frozen training-run datasets (`scripts/make_data.py`).

| file       | ops | format   | labels  | notes |
|------------|-----|----------|---------|-------|
| `D_algo`   | + - | nl       | correct | natural-language add/sub |
| `D_inst`   | *   | operator | random  | operator-format mult, random labels |
| `D_target` | + - | operator | correct | operator-format add/sub |
| `probe`    | + - | operator | correct | held-out eval set (64/cell) |

Use `show(...)` below to filter by **format** (`nl` / `operator`) and **op** (`add`/`sub`/`mult` or `+`/`-`/`*`).

In [ ]:
import subprocess, sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 120)

# ---- config: edit these ----
SCALE = 'pilot'          # 'pilot' (10k/dataset) or 'full' (1M/dataset)
SEED = 20260717

# find repo root by walking up until we see the geode/ package
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / 'geode').is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

# point DATA_DIR at your own parquet dir to view the real run instead
DATA_DIR = REPO_ROOT / 'experiments' / 'training-run' / 'notebooks' / 'data' / SCALE

print('repo root :', REPO_ROOT)
print('data dir  :', DATA_DIR)

In [ ]:
# generate the pilot if the parquet files are not there yet (CPU-only, no GPU/budget gate)
FILES = ['D_algo', 'D_inst', 'D_target', 'probe']
missing = [f for f in FILES if not (DATA_DIR / (f + '.parquet')).exists()]

if missing:
    print('generating', SCALE, 'dataset (missing:', missing, ')...')
    script = REPO_ROOT / 'experiments' / 'training-run' / 'scripts' / 'make_data.py'
    subprocess.run(
        [sys.executable, str(script), '--scale', SCALE, '--out', str(DATA_DIR), '--seed', str(SEED)],
        check=True,
    )
else:
    print('found existing parquet in', DATA_DIR)

In [ ]:
# load all four files into one combined frame
df = pd.concat([pd.read_parquet(DATA_DIR / (n + '.parquet')) for n in FILES], ignore_index=True)
df['op_word'] = df['op'].map({'+': 'add', '-': 'sub', '*': 'mult'})

print('total rows:', len(df))
df.groupby(['dataset', 'format', 'label_mode', 'op']).size().rename('rows').reset_index()

## Filter

`show(fmt=..., op=..., dataset=..., n=...)` — every argument is optional; omit one to keep it unfiltered.
- `fmt`: `'nl'` or `'operator'`
- `op`: `'add'`/`'sub'`/`'mult'` (or `'+'`/`'-'`/`'*'`)
- `dataset`: `'D_algo'`, `'D_inst'`, `'D_target'`, `'probe'`

In [ ]:
OP_ALIASES = {'add': '+', 'sub': '-', 'subtract': '-', 'minus': '-',
              'mult': '*', 'multiply': '*', 'times': '*',
              '+': '+', '-': '-', '*': '*'}

def show(fmt=None, op=None, dataset=None, n=10, data=None):
    'Filter the combined frame by format / op / dataset and return the first n matching rows.'
    out = df if data is None else data
    if fmt is not None:
        out = out[out['format'] == fmt]
    if op is not None:
        out = out[out['op'] == OP_ALIASES[op]]
    if dataset is not None:
        out = out[out['dataset'] == dataset]
    print('matched', len(out), 'rows for', dict(fmt=fmt, op=op, dataset=dataset))
    cols = ['dataset', 'a', 'op', 'b', 'cell', 'format', 'label_mode',
            'true_answer', 'shown_answer', 'full_text']
    return out[cols].head(n)

In [ ]:
# natural-language rows
show(fmt='nl', n=5)

In [ ]:
# multiplication rows (operator format, random labels)
show(op='mult', n=5)

In [ ]:
# operator-format subtraction, correct labels
show(op='sub', fmt='operator', n=5)

## Distribution across digit-cells

How many rows land in each `x_digits x y_digits` cell, per dataset.

In [ ]:
counts = df.groupby(['cell', 'dataset']).size().unstack('dataset').fillna(0)
counts = counts.reindex(sorted(counts.index, key=lambda c: (int(c[0]), int(c[2]))))
ax = counts.plot(kind='bar', figsize=(12, 4))
ax.set_xlabel('cell (x_digits x y_digits)')
ax.set_ylabel('rows')
ax.set_title('rows per digit-cell by dataset (' + SCALE + ')')
plt.tight_layout()
plt.show()